# Lab 1, part B: the agent

Lab 1 (A+B) Costs: $0.099

Part A built an MCP server and proved it from Claude Code. This part hands the same server
to an Agent SDK consumer, running inside this notebook, and then takes it apart: why the
agent picks the wrong tool, what actually stops a tool call, and what a hook can guarantee
that a prompt cannot.

This part does call a model. Every run below caps its turns and its spend.

Setup, once, in the `code/` directory above this one: copy `.env.example` to `.env` and read
the notes at the top of it. On a Claude subscription you leave the credential lines blank and
run `claude` once to sign in; on API billing you put a key in `ANTHROPIC_API_KEY`. Every lab
reads that same file, and the cell below prints which of the two it is about to use.

In [ ]:
import json
import os
from pathlib import Path

from dotenv import load_dotenv

LAB = Path.cwd()
CODE = LAB.parent
WORKSPACE = LAB / "workspace"
SERVERS = WORKSPACE / "servers"

if not (CODE / "pyproject.toml").exists():
    raise SystemExit(
        f"Run this notebook from its own folder inside labs/code. The working "
        f"directory is {LAB}."
    )

ENV_FILE = CODE / ".env"
if not ENV_FILE.exists():
    raise SystemExit(f"No {ENV_FILE}. Copy .env.example to .env and read the notes at the top of it.")

load_dotenv(ENV_FILE)

# A name left blank in .env still reaches the environment, as an empty string. An empty
# ANTHROPIC_API_KEY fails the run rather than falling through to a login, so drop the blanks
# and let the credential chosen below be one that is really set.
for name in ("ANTHROPIC_API_KEY", "CLAUDE_CODE_OAUTH_TOKEN"):
    if not os.environ.get(name):
        os.environ.pop(name, None)

# The Agent SDK runs the claude binary rather than calling the API itself, so it takes the
# credential that binary takes, in that binary's order of preference: key first, then token,
# then the login claude saved when you signed in.
if os.environ.get("ANTHROPIC_API_KEY"):
    CREDENTIAL = "ANTHROPIC_API_KEY from .env, billed to your API account"
elif os.environ.get("CLAUDE_CODE_OAUTH_TOKEN"):
    CREDENTIAL = "CLAUDE_CODE_OAUTH_TOKEN from .env, drawn from your subscription"
else:
    CREDENTIAL = "the login claude saved, drawn from your subscription"

if not os.environ.get("LAB_MODEL"):
    raise SystemExit(f"{ENV_FILE} has no value for LAB_MODEL.")
MODEL = os.environ["LAB_MODEL"]

if not WORKSPACE.is_dir():
    raise SystemExit(f"No {WORKSPACE}. Run part A first: it writes the workspace this part uses.")

print(f"workspace  {WORKSPACE}")
print(f"model      from LAB_MODEL in {ENV_FILE}")
print(f"credential {CREDENTIAL}")

## 1. One agent, one server

`query()` runs the same engine Claude Code runs, inside this process. Four options carry
the whole setup:

- `mcp_servers` connects the part A server over stdio, the same way `.mcp.json` did.
- `cwd` is the directory the built-in tools operate in, so the agent's world is the workspace.
- `setting_sources=[]` loads nothing from disk: no `CLAUDE.md`, no project settings, nothing
  from your own machine. Everything the agent has is in this cell, which is what makes the
  next few runs worth comparing.
- `max_turns` and `max_budget_usd` are the two limits worth setting on every run.

The helper collects the tool calls as they happen, because which tool got picked is the
thing this part measures.

The question below is one the ticket queue can answer and nothing else can. Your server is
connected and its tool is approved. Watch what the agent reaches for anyway.

These three sections give the agent search tools only, no shell. With `Bash` available it
just reads the file and answers, which is perfectly sensible and tells you nothing about how
it chooses between tools.

In [ ]:
from claude_agent_sdk import (
    AssistantMessage, ClaudeAgentOptions, ResultMessage, ToolUseBlock, query,
)

DEVTOOLS = {
    "type": "stdio",
    "command": "uv",
    "args": ["run", "--project", str(CODE), "python", str(SERVERS / "devtools_server.py")],
}


# A claude.ai login carries your organisation's connectors into the session, and
# setting_sources=[] does not exclude them: a connector is not a filesystem setting.
# Left on, a student signed in to a subscription gets MCP tools this lab never defined,
# in the middle of a lab about which tool the agent reaches for. A key does not load
# them, and neither does a setup-token, so this is the one line that makes the run the
# same on every credential.
NO_CONNECTORS = {"ENABLE_CLAUDEAI_MCP_SERVERS": "false"}


async def run(prompt, **overrides):
    options = ClaudeAgentOptions(
        model=MODEL,
        cwd=str(WORKSPACE),
        setting_sources=[],
        env=NO_CONNECTORS,
        max_turns=8,
        max_budget_usd=0.10,
        **overrides,
    )
    calls, result = [], None
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                # ToolSearch is how the SDK loads a tool's schema on demand. It is
                # plumbing, not a choice the model made, so it is not part of the count.
                if isinstance(block, ToolUseBlock) and block.name != "ToolSearch":
                    calls.append((block.name, block.input))
        elif isinstance(message, ResultMessage):
            result = message
    return calls, result


def names(calls):
    runs = []
    for name, _ in calls:
        if runs and runs[-1][0] == name:
            runs[-1][1] += 1
        else:
            runs.append([name, 1])
    return [n if c == 1 else f"{n} x{c}" for n, c in runs]


def say(result, limit=500):
    text = (result.result or "(no final text)").strip()
    print("  agent says:")
    for line in text[:limit].splitlines():
        print(f"    {line}")
    if len(text) > limit:
        print("    ...")


def show(calls):
    for name, arguments in calls:
        print(f"    {name}({json.dumps(arguments)[:72]})")


QUESTION = "has anyone reported customers being charged twice?"

# Search tools only. A shell would let the model cat the file and answer without choosing
# a tool at all, which is a different lesson from the one these three sections teach.
SEARCH_ONLY = ["Grep", "Glob", "Read"]

calls, result = await run(
    QUESTION,
    mcp_servers={"devtools": DEVTOOLS},
    tools=SEARCH_ONLY,
    allowed_tools=["mcp__devtools__lookup"] + SEARCH_ONLY,
)
print(f"tools called: {names(calls)}")
show(calls)
say(result)
print(f"cost: ${result.total_cost_usd:.4f}, turns: {result.num_turns}")

## 2. A second server, and it gets worse

Tools from every configured server are offered to the model at once. That is the feature,
and it is where this goes wrong.

The knowledge base server below is not badly written. Its description is confident and
specific, and it is exactly the kind of thing a neighbouring team ships happily. Read what
it claims to cover before you run the cell.

Then look at what the agent picks, and notice that it is not being careless. It is being
obedient. One tool said `Look things up.` The other said it covers support tickets and
customer reports. Given only those two sentences it chose reasonably and answered wrongly,
and your server never got a look in.

In [ ]:
WIKI_PY = '''
# The neighbouring team's knowledge base server.

import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP
from pydantic import Field

DATA = Path(__file__).resolve().parent.parent / "data"
mcp = FastMCP("wiki")


@mcp.tool(description=(
    "Search everything the team has written down, including support tickets, customer "
    "reports, incidents, documents and architecture decisions."
))
def search_knowledge_base(query: str = Field(description="Search text")) -> dict:
    hits = []
    for path in sorted(DATA.glob("adr/*.md")):
        text = path.read_text()
        if query.lower() in text.lower():
            hits.append({"page": path.stem, "text": text[:300]})
    return {"query": query, "matched": len(hits), "results": hits}


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

(SERVERS / "wiki_server.py").write_text(WIKI_PY.lstrip("\n"), encoding="utf-8")

WIKI = {
    "type": "stdio",
    "command": "uv",
    "args": ["run", "--project", str(CODE), "python", str(SERVERS / "wiki_server.py")],
}

BOTH = {"devtools": DEVTOOLS, "wiki": WIKI}
OPEN_TOOLS = [
    "mcp__devtools__lookup", "mcp__devtools__service_catalogue",
    "mcp__devtools__architecture_decisions", "mcp__wiki__search_knowledge_base",
] + SEARCH_ONLY

print("the two descriptions competing for this question:")
print("  mcp__devtools__lookup             Look things up.")
print("  mcp__wiki__search_knowledge_base  Search everything the team has written down,")
print("                                    including support tickets, customer reports...")
print()

before, result = await run(QUESTION, mcp_servers=BOTH, tools=SEARCH_ONLY,
                           allowed_tools=OPEN_TOOLS)
print(f"question: {QUESTION}")
print(f"tools called: {names(before)}")
show(before)
say(result)
before_result = result
print(f"cost: ${result.total_cost_usd:.4f}")

## 3. Three causes, three fixes

Misrouting is not one problem, so it does not have one fix. The server written in part A has
three different faults, and each needs its own intervention.

**Describe.** `Look things up.` is not a description, it is a shrug. A description says what
the tool is for, when to reach for it, when not to, what goes in and what comes back.
Frank's rule for this is blunt: describe or die.

**Rename.** `lookup` tells nobody anything. `search_tickets` claims its territory in one
word, and the name is part of the description surface whether you meant it to be or not.

**Disclaim.** This is the one people miss, and here it is the actual cause. The knowledge
base tool wins by claiming support tickets it does not hold. A good description says what
the tool is not for, so `search_wiki_pages` now states outright that tickets live elsewhere.
A tool that over-claims steals calls it cannot serve.

**Split.** `architecture_decisions` does two jobs: list what exists, and fetch one. A tool
that does two things cannot be described crisply as either, so it becomes
`list_architecture_decisions` and `read_architecture_decision`.

There is a fourth cause, and it is the one to rule out first, because it is not the tool's
fault at all. A system prompt or a `CLAUDE.md` that names a tool by keyword, or that says
something like "prefer grep for searching", overrides a perfectly good description. That is
what `setting_sources=[]` is keeping out of these runs: if a colleague reports misrouting
you cannot reproduce, their project configuration is the first place to look.

The fixed servers go in new files so the originals stay readable beside them.

In [ ]:
FIXED_DEVTOOLS = '''
# Developer productivity tools. Descriptions rewritten, the ADR tool split in two.

import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP
from pydantic import Field

from errors import tool_error

DATA = Path(__file__).resolve().parent.parent / "data"
mcp = FastMCP("devtools")


@mcp.tool(description=(
    "Search the Fernhill support ticket queue by free text. Use this for anything a person "
    "reported: problems, incidents, bugs, customer complaints, and whether a ticket is "
    "still open. Do not use it for documentation or source code. Returns matching tickets "
    "with id, status, owning service, title and body, and a count. A query that matches "
    "nothing returns an empty list, not an error."
))
def search_tickets(query: str = Field(description="Free text, such as 'duplicate charges'")) -> dict:
    tickets = json.loads((DATA / "tickets.json").read_text())
    hits = [t for t in tickets if query.lower() in (t["title"] + " " + t["body"]).lower()]
    return {"query": query, "matched": len(hits), "results": hits}


@mcp.tool(description=(
    "Look up one Fernhill service by its exact name, to find who owns it, who is on call, "
    "and which file it starts from. Use this when you need the owner or the entrypoint of a "
    "named service. Do not use it to search for a service by description. "
    "Returns one service record, or a validation error listing the names that exist."
))
def service_catalogue(name: str = Field(description="Exact service name, such as 'refunds'")) -> dict:
    services = json.loads((DATA / "services.json").read_text())
    for service in services:
        if service["name"] == name:
            return service
    known = ", ".join(s["name"] for s in services)
    raise tool_error("validation", f"No service called {name!r}. Expected one of: {known}.",
                     "Call again with one of the listed names.")


@mcp.tool(description=(
    "List the ids of every architecture decision record. Takes no arguments. "
    "Call this first when you do not already know which record you need, then read one."
))
def list_architecture_decisions() -> dict:
    return {"available": [p.stem for p in sorted(DATA.glob("adr/ADR-*.md"))]}


@mcp.tool(description=(
    "Read the full text of one architecture decision record by its id, such as 'ADR-0005'. "
    "Use this for standing engineering decisions and the reasoning behind them. "
    "Call list_architecture_decisions first if you do not know the id."
))
def read_architecture_decision(adr_id: str = Field(description="An id such as 'ADR-0005'")) -> dict:
    for path in sorted(DATA.glob("adr/ADR-*.md")):
        if path.stem.startswith(adr_id):
            return {"id": path.stem, "text": path.read_text()}
    raise tool_error("validation", f"No decision record {adr_id!r}.",
                     "Call list_architecture_decisions to see the ids that exist.")


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

FIXED_WIKI = '''
# The knowledge base, renamed and narrowed so it stops claiming tickets.

from pathlib import Path

from mcp.server.fastmcp import FastMCP
from pydantic import Field

DATA = Path(__file__).resolve().parent.parent / "data"
mcp = FastMCP("wiki")


@mcp.tool(description=(
    "Search the team's long form wiki pages, which are prose written by engineers to explain "
    "standing decisions. Use this for background and rationale. Do not use it for support "
    "tickets or customer reports, which live in the ticket queue."
))
def search_wiki_pages(query: str = Field(description="Search text")) -> dict:
    hits = []
    for path in sorted(DATA.glob("adr/*.md")):
        text = path.read_text()
        if query.lower() in text.lower():
            hits.append({"page": path.stem, "text": text[:300]})
    return {"query": query, "matched": len(hits), "results": hits}


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

(SERVERS / "devtools_fixed.py").write_text(FIXED_DEVTOOLS.lstrip("\n"), encoding="utf-8")
(SERVERS / "wiki_fixed.py").write_text(FIXED_WIKI.lstrip("\n"), encoding="utf-8")

FIXED = {
    "devtools": {**DEVTOOLS, "args": ["run", "--project", str(CODE), "python",
                                      str(SERVERS / "devtools_fixed.py")]},
    "wiki": {**WIKI, "args": ["run", "--project", str(CODE), "python",
                              str(SERVERS / "wiki_fixed.py")]},
}
FIXED_TOOLS = [
    "mcp__devtools__search_tickets", "mcp__devtools__service_catalogue",
    "mcp__devtools__list_architecture_decisions", "mcp__devtools__read_architecture_decision",
    "mcp__wiki__search_wiki_pages",
] + SEARCH_ONLY

after, result = await run(QUESTION, mcp_servers=FIXED, tools=SEARCH_ONLY,
                          allowed_tools=FIXED_TOOLS)

print(f"question: {QUESTION}")
print(f"  before: {names(before)}")
print(f"  after:  {names(after)}")
print("before, the agent says:")
say(before_result)
print("after, the agent says:")
say(result)
print(f"cost: ${result.total_cost_usd:.4f}")
print()
print("Look at the counts, not just the names. Before the fix the agent can spend call after")
print("call on the tool that claimed the territory, rephrasing the query, never finding the")
print("ticket, and never asking the server that holds it. This pairing was measured at five")
print("out of five wrong before and five out of five right after. Routing is still the")
print("model's decision though, so anything that has to hold every time belongs in code.")

## 4. The built-in tools, chosen on purpose

The agent also has tools nobody had to write. `Grep` searches content, `Glob` matches paths,
`Read` and `Write` work on whole files, `Edit` replaces a unique piece of text, and `Bash`
runs a command.

Two things to watch in this run, both of them planted in part A.

`charge_card` is re-exported twice, as `take_payment` and as `process_payment`. A single
grep for one name finds a third of the answer, so the agent has to find the aliases first.

And `max_retries = 3` appears twice, identically. `Edit` needs its anchor to be unique, so
the obvious edit fails, and the way out is a longer anchor or a read and a write.

`tools` names the tools that exist for this run. That is a different thing from
`allowed_tools`, which is the subject of the next section.

In [ ]:
# Re-runnable: put the retry count back before asking for the change, so the cell still
# demonstrates something on a second pass.
refunds = WORKSPACE / "shop" / "refunds.py"
refunds.write_text(refunds.read_text().replace("max_retries = 5", "max_retries = 3"))

TASK = (
    "In shop/, find every place a card is actually charged, including through re-exported "
    "names. Then in shop/refunds.py raise the retry count in send_refund from 3 to 5, and "
    "leave send_reversal alone."
)

calls, result = await run(
    TASK,
    tools=["Grep", "Glob", "Read", "Edit", "Write"],
    allowed_tools=["Grep", "Glob", "Read", "Edit", "Write"],
    permission_mode="acceptEdits",
)
print(f"tools called: {names(calls)}")
say(result)
print(f"cost: ${result.total_cost_usd:.4f}, turns: {result.num_turns}")
print()
print((WORKSPACE / "shop" / "refunds.py").read_text())

## 5. Availability, approval, and the decision at the door

Three different things, and they are easy to run together.

`tools` is **availability**: what exists for this run at all.
`allowed_tools` is **approval**: what runs without stopping to ask.
`permission_mode` is the **posture** for everything that was not pre-approved, and `dontAsk`
is the one for unattended runs, because a prompt nobody is there to answer is a hang.

Here is where it gets interesting. It is natural to read an allowlist plus `dontAsk` as a
boundary: these tools and no others. The next two cells test that, and the exam's framing
and the measured behaviour do not agree.

In [ ]:
calls, result = await run(
    "Run `ls -la` in the current directory and tell me what you see.",
    permission_mode="dontAsk",
    allowed_tools=["Read"],
)
print("allowed_tools = ['Read'], permission_mode = 'dontAsk', and Bash is neither.")
print()
print(f"tools called:       {names(calls)}")
show(calls)
say(result)
print(f"permission denials: {result.permission_denials}")
print()
print("Read the two lines above rather than taking my word for it. If Bash is in the")
print("list and the denials are empty, the allowlist did not stop anything.")

In [ ]:
calls, result = await run(
    "Delete the build directory by running exactly this and nothing else: rm -rf build",
    permission_mode="dontAsk",
    allowed_tools=["Read", "Bash"],
    disallowed_tools=["Bash(rm *)"],
)
print("Same mode. This time Bash is allowed, and a deny rule names the command.")
print()
print(f"tools called:       {names(calls)}")
show(calls)
say(result)
print(f"permission denials: {result.permission_denials}")
print()
attempted = [a.get("command", "") for n, a in calls if n == "Bash"]
if any(c.strip().startswith("rm") for c in attempted):
    print("The rm was attempted. Whether it ran is what the denials line tells you.")
else:
    print("The model did not attempt the rm at all, so the rule had nothing to match.")
    print("That is worth seeing too: a deny rule only fires on a call that is made.")

So the allowlist approves, it does not restrict. What actually stops a call is leaving the
tool out of `tools`, or naming it in `disallowed_tools`, and deny rules are evaluated before
the mode, which is why they hold even in the permissive ones.

Two honest notes about the cell above. A deny rule only fires on a call the model actually
makes, and a model asked to delete something may reasonably look before it leaps, so you
may see it run `ls` instead and the denials stay empty. And the cell before it is the one
that carries the lesson either way: an unapproved `Bash` running under `dontAsk` is the
whole point, and it does not depend on the model co-operating.

Answer an exam item about `allowedTools` with the restriction it describes, because in an
unattended run an unapproved tool does not reach a human. Then build the boundary out of
`tools` and deny rules, because that is the half that holds.

## 6. A hook is a guarantee

Everything so far has been persuasion: a description that makes a tool attractive, an
allowlist that makes it cheap. A hook is different. `PreToolUse` runs in your process,
before the call goes out, and what it returns is not advice.

Two of them below. The first records every call and changes nothing, which is how you get an
audit trail. The second refuses one, and the reason it returns goes back to the model, so the
agent can adapt rather than simply fail.

In [ ]:
from claude_agent_sdk import HookMatcher

seen = []


async def record(input_data, tool_use_id, context):
    if input_data["tool_name"] != "ToolSearch":  # plumbing, as above
        seen.append((input_data["tool_name"], input_data.get("tool_input", {})))
    return {}


calls, result = await run(
    "What is decision ADR-0005 about, and who is on call for the refunds service?",
    mcp_servers=FIXED,
    allowed_tools=FIXED_TOOLS,
    hooks={"PreToolUse": [HookMatcher(hooks=[record])]},
)

print(f"tools called: {names(calls)}")
say(result)
print()
for name, arguments in seen:
    print(f"  {name}({json.dumps(arguments)[:70]})")

In [ ]:
async def refuse_wiki(input_data, tool_use_id, context):
    return {
        "hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": (
                "The wiki is being reindexed. Use the ticket queue and the decision records."
            ),
        }
    }


seen.clear()
calls, result = await run(
    "Search the wiki for anything about duplicate charges, then tell me what you found.",
    mcp_servers=FIXED,
    allowed_tools=FIXED_TOOLS,
    hooks={
        "PreToolUse": [
            HookMatcher(matcher="mcp__wiki__.*", hooks=[refuse_wiki]),
            HookMatcher(hooks=[record]),
        ]
    },
)

print(f"tools called: {names(calls)}")
print(f"cost: ${result.total_cost_usd:.4f}")
print()
say(result, 400)

## What you built

Read this back against the diagram.

| Diagram | Where it was built |
|---|---|
| MCP server, three tools | Part A, sections 4 and 5 |
| Structured error, category and retryable and next action | Part A, sections 2 and 3 |
| Interactive Claude Code, project and user scope | Part A, sections 7 and 8 |
| Agent SDK consumer over the same server | Part B, section 1 |
| Tool routing by descriptions, describe and rename and split | Part B, sections 2 and 3 |
| Built-in tools | Part B, section 4 |
| Allowed tools, and what a boundary actually is | Part B, section 5 |
| PreToolUse hook, record and deny | Part B, section 6 |

The decision to carry out of this lab: given a report that an agent reached for the wrong
tool, say whether the fix is a description, a rename, a split, or not the tool's fault at
all. And given a run that must not touch something, reach for `tools` and a deny rule rather
than an allowlist.